# M7. 영화 리뷰 낱말 세기

네이버 영화 리뷰 5만 건에서 빈 리뷰 · 중복 · 정답이 엇갈린 글을 걸러 내고, 형태소 분석기 없이 낱말을 잘라 DuckDB 로 긍정 · 부정 리뷰에 유난히 많이 나오는 낱말을 로그 오즈로 찾습니다.

- 단계: 기초 미니 프로젝트 · 비정형 · 예상 반나절~하루
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: Naver Sentiment Movie Corpus(NSMC, Lucy Park) — ratings_test.txt 5만 건, CC0 1.0(퍼블릭 도메인)

**빈칸 노트북** — 코드 속 `___①___` 자리 12곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/M7/M7_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

리뷰 글은 대표적인 **비정형 데이터**입니다. 칸이 정해진 표가 아니라 사람이 쓴 문장이라, 숫자로 다루려면 먼저 「낱말」이라는 단위로 잘라야 합니다.
이 프로젝트에서는 영화 리뷰 5만 건을 잘라 **「긍정 리뷰에만 유난히 많은 낱말」과 「부정 리뷰에만 유난히 많은 낱말」**을 찾습니다.

```text
NSMC 리뷰(TSV) → pandas 정제(빈 글 · 중복 · 정답이 엇갈린 글) → 낱말 자르기(조사 떼기 · 불용어)
  → DuckDB 집계(긍정/부정 건수 · 비율 · 로그 오즈) → Parquet · 그래프
```

NSMC 는 리뷰마다 **정답 감성(1 긍정 · 0 부정)**이 붙어 있어서, 낱말마다 「긍정 쪽에 얼마나 치우쳤나」를 바로 잴 수 있습니다.
다 끝내면 다음을 말할 수 있습니다.

- 5만 건 가운데 분석에 쓴 리뷰가 몇 건이고, 무엇을 왜 뺐는지
- 불용어를 빼지 않으면 왜 「영화」가 1위가 되는지
- 「긍정 비율」만으로 줄 세우면 왜 틀리는지, 로그 오즈가 그것을 어떻게 고치는지

## 1단계 · 준비 — 라이브러리와 데이터

Colab 에는 pandas 와 DuckDB 가 이미 깔려 있습니다. 데이터는 공개 자료실에서 받습니다.

데이터는 네이버 영화 리뷰 말뭉치 NSMC(Naver Sentiment Movie Corpus)의 평가용 파일 `ratings_test.txt` 입니다(P6 에서 쓴 것과 같은 파일).
원작자(Lucy Park)가 **CC0 1.0**(퍼블릭 도메인)으로 공개해서 자유롭게 고쳐 쓰고 다시 나눌 수 있습니다. 내용은 원본 그대로이고 gzip 으로 압축만 했습니다.

In [ ]:
import os, re, shutil, urllib.request
from collections import Counter
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

path = fetch("nsmc_ratings_test.txt.gz")
print(path, round(os.path.getsize(path) / 1e6, 1), "MB")

> **확인 포인트** — `data/nsmc_ratings_test.txt.gz 2.1 MB`.

파일 이름은 `.txt` 지만 속은 **탭으로 칸을 나눈 표(TSV)**입니다. 쉼표로 나뉜 CSV 인 줄 알고 그냥 읽으면 어떻게 될까요?

In [ ]:
try:
    pd.read_csv(path)
except pd.errors.ParserError as e:
    print("읽기 실패:", str(e)[:70])

리뷰 글 안에 쉼표가 들어 있어서, 어떤 줄은 칸이 1개 · 어떤 줄은 2개로 잘려 실패합니다. 칸 나누개(구분자)를 탭으로 알려 줍니다.

In [ ]:
reviews = pd.read_csv(path, sep="___①___", dtype={"document": str})   # ① 탭 문자
print(reviews.shape, reviews.columns.tolist())
print(reviews["label"].value_counts().to_dict())
reviews.head(3)

> **확인 포인트** — `(50000, 3)` — 열은 `id` · `document`(리뷰 글) · `label`(1 긍정, 0 부정), 긍정 25,173 · 부정 24,827 건입니다.

In [ ]:
assert reviews.shape == (50000, 3) and reviews["label"].value_counts().to_dict() == {1: 25173, 0: 24827}

> **Aebon's Tip** — 원본은 github.com/e9t/nsmc 에 있습니다. 학습용 `ratings_train.txt`(15만 건)도 같은 형식이라, 파일 이름만 바꾸면 이 노트북이 그대로 돕니다.

## 2단계 · 정제 — 빈 글 · 중복 · 정답이 엇갈린 글

낱말을 자르기 전에 글부터 고릅니다. 먼저 정제하지 않은 채 정규식으로 낱말을 찾아 봅니다.

In [ ]:
try:
    n = sum(len(re.findall(r"[가-힣]{2,}", t)) for t in reviews["document"])
except TypeError as e:
    print("실패:", e)

`document` 에 빈 값(`NaN`, 숫자형 float)이 섞여 있어 정규식이 멈춥니다. 몇 건인지 세고 뺍니다. 앞뒤 공백도 지워 둡니다 — 공백만 다른 글을 같은 글로 보기 위해서입니다.

In [ ]:
print("빈 글:", reviews["document"].isna().sum())
r = reviews.___②___(subset=["document"]).copy()   # ② 빈 값이 있는 행 빼기
r["text"] = r["document"].str.___③___()   # ③ 앞뒤 공백 지우기
print("남은 글:", len(r), "· 공백만 있는 글:", (r["text"] == "").sum())

> **확인 포인트** — 빈 글 **3건**을 빼고 49,997건이 남습니다. 공백만 있는 글은 없습니다.

In [ ]:
assert len(r) == 49997 and (r["text"] == "").sum() == 0

다음은 **중복**입니다. 같은 글을 여러 번 세면 그 글의 낱말이 부풀려집니다. 글과 정답이 모두 같은 행은 하나만 남깁니다.

In [ ]:
d1 = r.___④___(subset=["text", "label"])   # ④ 겹치는 행 하나만 남기기
print("완전히 같은 행:", len(r) - len(d1), "· 남은 글:", len(d1))

그런데 글은 같은데 **정답이 서로 다른** 경우가 있습니다. 어떤 글인지 봅니다.

In [ ]:
n_label = d1.groupby("text")["label"].___⑤___("nunique")   # ⑤ 그룹 값을 행마다 붙이기
conflict = d1[n_label > 1].sort_values(["text", "label"])
print("정답이 엇갈린 글:", conflict["text"].nunique(), "가지 ·", len(conflict), "행")
conflict.head(6)[["text", "label"]]

`.` · `...` · `?` 처럼 감성을 판단할 근거가 없는 글입니다. 같은 글에 서로 다른 정답이 붙어 있으면 「이 낱말은 긍정 쪽이다」를 셀 수 없으니 통째로 뺍니다.

In [ ]:
clean = d1[n_label == 1].reset_index(drop=True)
print("정제 후:", len(clean), clean["label"].value_counts().to_dict())

> **확인 포인트** — 완전히 같은 행 **793건**, 정답이 엇갈린 글 **47가지(94행)**를 빼고 **49,110건**(긍정 24,687 · 부정 24,423)이 남습니다.

In [ ]:
assert len(r) - len(d1) == 793 and conflict["text"].nunique() == 47 and len(conflict) == 94
assert len(clean) == 49110 and clean["label"].value_counts().to_dict() == {1: 24687, 0: 24423}

> **자주 겪는 오류** — `drop_duplicates(subset=["text"])` 처럼 글만 기준으로 하면, 정답이 엇갈린 글이 **먼저 나온 정답 하나로** 남습니다. 어느 쪽이 남을지가 파일 순서에 달린 정제는 피하세요.

## 3단계 · 낱말 자르기 — 형태소 분석기 없이

M4 7단계에서 쓴 간단한 규칙을 그대로 씁니다.

1. 한글 2글자 이상, 또는 영문 2글자 이상을 낱말로 봅니다(정규식). 영문은 소문자로 맞춥니다(`OST` · `ost` 를 같게).
2. 낱말 끝의 조사(`의` · `를` · `에서` …)를 한 번 떼어 냅니다. 남는 글자가 2글자 이상일 때만 뗍니다.
3. 불용어(뜻을 가르는 데 도움이 안 되는 낱말)를 뺍니다 — 이 단계에서는 **일부러 아직 빼지 않습니다.**

In [ ]:
JOSA = ("으로", "에서", "에게", "부터", "까지", "의", "를", "을", "에", "은", "는", "이", "가", "로", "와", "과", "도")

def words(text, stop=frozenset()):
    out = []
    for w in re.findall(r"[가-힣]{2,}|[A-Za-z]{2,}", text):
        if w.isascii():
            w = w.___⑥___()   # ⑥ 영문을 소문자로
        else:
            for j in JOSA:
                if w.___⑦___(j) and len(w) - len(j) >= 2:   # ⑦ 끝이 조사로 끝나면
                    w = w[:-len(j)]
                    break
        if w not in stop:
            out.append(w)
    return out

sample = clean.loc[clean["id"] == 7898805, "text"].item()      # 실제 긍정 리뷰 한 건
print(sample)
print(words(sample))

> **확인 포인트** — 「음악이 주가 된, 최고의 음악영화」 → `['음악', '주가', '최고', '음악영화']`.
> `음악이` · `최고의` 는 조사가 떨어졌지만, `주가`(주 + 가)는 남는 글자가 1글자라 그대로 남았습니다. `음악영화` 처럼 붙여 쓴 말도 나뉘지 않습니다. 간단한 규칙의 한계입니다.

In [ ]:
assert sample == "음악이 주가 된, 최고의 음악영화" and words(sample) == ["음악", "주가", "최고", "음악영화"]

불용어 없이 전체를 세 봅니다.

In [ ]:
c_all = Counter()
for t in clean["text"]:
    c_all.update(words(t))
c_all.most_common(6)

> **확인 포인트** — 1위 **영화 10,240번**, 이어서 너무 2,936 · 정말 2,904 · 진짜 2,416 입니다.
> 영화 리뷰에 「영화」가 많은 것은 당연하고, 「너무 · 정말 · 진짜」는 좋은 말에도 나쁜 말에도 붙는 힘줌말입니다. 이대로는 아무것도 알려 주지 않습니다.

In [ ]:
assert c_all.most_common(4) == [("영화", 10240), ("너무", 2936), ("정말", 2904), ("진짜", 2416)]

위 순위를 보고 불용어 목록을 만듭니다. 기준은 「긍정 · 부정 어느 쪽에서도 위에 있고, 뜻을 가르지 않는 말」입니다 — 영화 그 자체, 힘줌말, 가리키는 말, 「보다」가 꼴을 바꾼 말.

In [ ]:
STOP = {"영화", "너무", "정말", "진짜", "완전", "그냥",
        "이런", "이건", "이거", "이게", "이렇게", "그리고", "내가",
        "보고", "봤는데", "보는", "보다"}

rows, empty = [], 0
for rid, text, label in zip(clean["id"], clean["text"], clean["label"]):
    ws = words(text, ___⑧___)   # ⑧ 뺄 낱말 목록
    empty += len(ws) == 0
    rows += [(rid, label, w) for w in ws]
tok = pd.DataFrame(rows, columns=["id", "label", "word"])
print("낱말:", len(tok), "· 서로 다른 낱말:", tok["word"].nunique(), "· 낱말이 하나도 없는 리뷰:", empty)
print(Counter(tok["word"]).most_common(5))

> **확인 포인트** — 낱말 **330,499개**, 서로 다른 낱말 104,304개입니다. 1위는 이제 **평점 1,595번**, 이어서 연기 · 스토리 · 최고 · 드라마입니다.
> 낱말이 하나도 남지 않은 리뷰가 340건 있습니다(「굳 ㅋ」 처럼 한 글자 · 자음뿐인 글). 그 리뷰는 낱말 표에 나타나지 않을 뿐 지우지는 않았습니다.

In [ ]:
assert len(tok) == 330499 and tok["word"].nunique() == 104304 and empty == 340
assert Counter(tok["word"]).most_common(1) == [("평점", 1595)]

> **Aebon's Tip** — `쓰레기영화` 는 「영화」를 불용어로 뺐는데도 남습니다. 붙여 쓴 말 안의 「영화」는 정규식이 한 낱말로 보기 때문입니다. 띄어쓰기가 엉망인 리뷰 글에서는 형태소 분석기(예: kiwipiepy)가 이런 것을 나눠 줍니다.

## 4단계 · DuckDB — 긍정 · 부정에 치우친 낱말

낱말 표(`tok`)는 한 줄에 낱말 하나인 「긴 표」입니다. DuckDB 에서 낱말마다 긍정 · 부정 건수를 `FILTER` 로 나눠 셉니다(3주차 14장).

In [ ]:
counts = duckdb.sql("""
    SELECT word,
           count(*) ___⑨___ AS pos,   -- ⑨ 긍정 리뷰만 세기
           count(*) FILTER (WHERE label = 0) AS neg
    FROM tok
    GROUP BY word
""").df()
print(len(counts), "낱말 · 긍정 낱말 합", counts["pos"].sum(), "· 부정 낱말 합", counts["neg"].sum())
assert len(counts) == 104304 and counts["pos"].sum() == 162798 and counts["neg"].sum() == 167701

가장 쉬운 지표는 **긍정 비율** = 긍정 ÷ (긍정 + 부정)입니다. 이것으로 줄을 세워 봅니다.

In [ ]:
naive = duckdb.sql("""
    SELECT word, pos, neg, pos / (pos + neg) AS pos_share
    FROM counts ORDER BY pos_share DESC, word LIMIT 5
""").df()
n_perfect = duckdb.sql("SELECT count(*) FROM counts WHERE neg = 0").fetchall()[0][0]
n_once = duckdb.sql("SELECT count(*) FROM counts WHERE neg = 0 AND pos = 1").fetchall()[0][0]
print("긍정 비율 1.0 인 낱말:", n_perfect, "· 그중 딱 한 번 나온 낱말:", n_once)
naive

> **확인 포인트** — 긍정 비율이 1.0(부정에 한 번도 안 나옴)인 낱말이 **41,602개**이고, 그중 **36,394개**는 딱 한 번 나온 낱말입니다.
> 맨 위 5개는 모두 딱 한 번 나온 영어 낱말(`abandoned` · `abba` …)입니다. 같은 비율이면 가나다순으로 놓았을 뿐, 1등에 아무 뜻이 없습니다.
> 한 번 나온 낱말은 우연히 긍정 리뷰에 있었을 뿐인데 「완벽한 긍정 낱말」이 됩니다. 비율만 보면 적게 나온 낱말이 언제나 맨 위로 옵니다.

In [ ]:
assert n_perfect == 41602 and n_once == 36394
assert (naive["pos_share"] == 1.0).all() and (naive["pos"] == 1).all()

그래서 두 가지를 고칩니다. ① 50번 이상 나온 낱말만 봅니다(하한). ② **로그 오즈**(log odds)를 씁니다.

```text
log_odds = ln( (pos + 1) / (P + 1) ) - ln( (neg + 1) / (N + 1) )
           P · N = 긍정 · 부정 리뷰의 전체 낱말 수
```

긍정 쪽에서 그 낱말이 차지하는 몫과 부정 쪽 몫의 **비를 로그로** 잰 값입니다. 0 이면 치우침 없음, 양수면 긍정, 음수면 부정 쪽입니다. 전체 낱말 수(P · N)로 나누니 긍정 · 부정 리뷰 양이 달라도 공정합니다.
먼저 `+1` 없이 계산해 봅니다.

In [ ]:
try:
    duckdb.sql("""
        WITH t AS (SELECT sum(pos) AS P, sum(neg) AS N FROM counts)
        SELECT word, ln(pos / P) - ln(neg / N) AS log_odds FROM counts, t ORDER BY log_odds DESC LIMIT 3
    """).df()
    print("통과 — 이 DuckDB 판은 ln(0) 을 -inf 로 돌려줍니다. 1위가 무한대가 되어 순위가 쓸모없어집니다.")
except duckdb.Error as e:
    print(type(e).__name__, "—", str(e).splitlines()[0][:70])

한쪽에 한 번도 안 나온 낱말(`neg = 0`)의 로그가 0 의 로그가 되어 멈춥니다. 모든 건수에 1 을 더하면(**더하기 1 평활**, add-one smoothing) 0 이 사라지고, 적게 나온 낱말은 0 쪽으로 끌려와 과장이 줄어듭니다.

In [ ]:
lo = duckdb.sql("""
    WITH t AS (SELECT sum(pos) AS P, sum(neg) AS N FROM counts)
    SELECT word, pos, neg, pos + neg AS n,
           round(pos / (pos + neg), 3) AS pos_share,
           round(ln((___⑩___) / (P + 1)) - ln((neg + 1) / (N + 1)), 3) AS log_odds   -- ⑩ 0 을 없애는 평활
    FROM counts, t
    WHERE ___⑪___   -- ⑪ 50번 이상 나온 낱말만
    ORDER BY log_odds DESC, word
""").df()
print(len(lo), "낱말 · 긍정 쪽", (lo["log_odds"] > 0).sum())
lo.head(10)

> **확인 포인트** — 50번 이상 나온 낱말 **771개**(긍정 쪽 425개). 긍정 1위는 **재밌었음(4.292)**, 이어서 명작이다 · 감동적이고 · 재밌었어요 · 지금봐 순입니다.
> 「재밌었음 · 재밌었어요 · 재밌어요 · 재미있어요」가 따로 세어졌습니다 — 말끝이 다르면 다른 낱말이 되는 것도 간단한 규칙의 한계입니다.

In [ ]:
assert len(lo) == 771 and (lo["log_odds"] > 0).sum() == 425
assert lo.iloc[0]["word"] == "재밌었음" and lo.iloc[0]["log_odds"] == 4.292
assert lo["word"].tolist()[:3] == ["재밌었음", "명작이다", "감동적이고"]

In [ ]:
lo.tail(6)

> **확인 포인트** — 부정 1위(맨 아래)는 **이딴(-4.436)**, 그 위가 최악(-4.327) · 최악이다 · 쓰레기영화 · 아깝다입니다.
> `최악` 은 긍정 리뷰에도 5번 나오는데(「최악의 상황들은 다 보여주면서도 인물들 감정을 넘 잘 살리신…」 같은 글) 467 대 5 라 부정 쪽 끝에 섭니다. 비율이 아니라 **건수와 비율을 함께** 보는 지표라서입니다.

In [ ]:
assert lo.iloc[-1]["word"] == "이딴" and lo.iloc[-1]["log_odds"] == -4.436
assert lo.set_index("word").loc["최악", ["pos", "neg"]].tolist() == [5, 467]

영화를 이루는 요소들은 어느 쪽에 치우쳤을까요? 같은 표에서 몇 낱말만 골라 봅니다.

In [ ]:
parts = lo[lo["word"].isin(["ost", "연기", "배우", "평점", "스토리", "감독", "cg"])].sort_values("log_odds", ascending=False)
parts[["word", "pos", "neg", "log_odds"]]

> **확인 포인트** — `ost`(1.510) · 연기(0.375)는 긍정 쪽, 배우(0.030)는 한가운데, 평점 · 스토리 · 감독 · `cg`(-1.143)는 부정 쪽입니다.
> 사람들은 음악 · 연기를 칭찬할 때, 이야기 · 연출 · 컴퓨터그래픽을 탓할 때 그 낱말을 꺼낸다고 읽을 수 있습니다. 「평점」이 부정 쪽인 것은 「평점믿고 봤는데.. 난 그저 지루해서 …」 처럼 높은 평점에 속았다는 글이 많기 때문입니다.

In [ ]:
assert parts["word"].tolist() == ["ost", "연기", "배우", "평점", "스토리", "감독", "cg"]

## 5단계 · 저장과 그래프

낱말 표와 결과 표를 Parquet 으로 저장합니다(3주차 13장). 낱말 표는 다른 분석(예: 리뷰 길이별 · 낱말 조합)의 재료가 되니 함께 남깁니다.

In [ ]:
tok.___⑫___("data/nsmc_tokens.parquet", index=False)   # ⑫ Parquet 으로 저장
duckdb.sql("COPY (SELECT * FROM lo) TO 'data/nsmc_word_log_odds.parquet' (FORMAT parquet)")
for f in ("data/nsmc_tokens.parquet", "data/nsmc_word_log_odds.parquet"):
    print(f, round(os.path.getsize(f) / 1e6, 2), "MB")

back = duckdb.sql("""
    SELECT count(*) AS n, count(DISTINCT id) AS reviews FROM 'data/nsmc_tokens.parquet'
""").fetchall()[0]
print("다시 읽은 낱말:", back[0], "· 리뷰:", back[1])
assert back == (330499, 49110 - 340)

> **확인 포인트** — 다시 읽은 낱말 330,499개 · 리뷰 48,770건(49,110 − 낱말 없는 340건)입니다.

그래프 두 개를 그립니다. 낱말 글자는 한글이라 그림에는 넣지 않고(Colab 기본 글꼴에 한글이 없습니다), 낱말은 위의 표로 읽습니다.

- 왼쪽: 낱말을 많이 나온 순으로 줄 세워 순위와 횟수를 둘 다 로그 눈금으로 — 불용어를 빼기 전과 뺀 뒤
- 오른쪽: 50번 이상 나온 771개 낱말의 횟수(로그)와 로그 오즈

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
before = np.array(sorted(c_all.values(), reverse=True))
after = np.array(sorted(Counter(tok["word"]).values(), reverse=True))
ax1.loglog(np.arange(1, len(before) + 1), before, label="no stopwords removed")
ax1.loglog(np.arange(1, len(after) + 1), after, label="stopwords removed")
ax1.set_title("Word frequency by rank (NSMC test, 49,110 reviews)")
ax1.set_xlabel("rank (log)")
ax1.set_ylabel("count (log)")
ax1.legend()

colors = np.where(lo["log_odds"] > 0, "tab:blue", "tab:red")
ax2.scatter(lo["n"], lo["log_odds"], s=10, alpha=.6, c=colors)
ax2.axhline(0, color="gray", lw=1)
ax2.set_xscale("log")
ax2.set_title("Log odds (positive vs negative), words with n >= 50")
ax2.set_xlabel("count (log)")
ax2.set_ylabel("log odds  (+ positive / - negative)")
for a in (ax1, ax2):
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()
print("맨 위 낱말 횟수 — 빼기 전:", before[0], "· 뺀 뒤:", after[0])
assert before[0] == 10240 and after[0] == 1595

> **확인 포인트** — 왼쪽 두 선은 오른쪽 아래로 곧게 내려갑니다 — 소수의 낱말이 아주 많이, 대부분의 낱말은 한두 번 나오는 모양(지프의 법칙)입니다. 불용어를 빼면 맨 앞(10,240 → 1,595)만 깎입니다.
> 오른쪽은 많이 나온 낱말일수록 0 근처에 모이고, 양 끝(±4)에는 50~500번 나온 감정 낱말이 섭니다. 아주 흔한 말은 양쪽 리뷰에 다 쓰이기 때문입니다.

## 6단계 · 더 해 보기

- 3단계의 `STOP` 에 「평점」을 더하면 4단계 결과가 어떻게 바뀌나요? 불용어는 결과를 보고 정하는 만큼, 왜 뺐는지를 보고서에 함께 적으세요.
- 「재밌」 · 「재미있」 으로 시작하는 낱말을 한 낱말 `재미있다` 로 합쳐(M4 의 `SAME` 사전처럼) 다시 세어 보세요. 긍정 1위가 바뀌나요?
- 두 낱말 묶음(바이그램, 예: `연기 최고`)을 세어 보세요. `tok` 을 `id` 로 묶어 이웃한 낱말을 이으면 됩니다.
- 낱말 자르기를 DuckDB 안에서도 해 보세요 — `unnest(regexp_extract_all(text, '[가-힣]{2,}'))` 가 한 줄에 낱말 하나씩 펼쳐 줍니다. 조사 떼기는 어떻게 옮길 수 있을까요?
- 이 로그 오즈 표는 P6(리뷰 감성 파이프라인)의 기준 모델이 어떤 낱말을 근거로 판단하는지 짐작하는 데 쓸 수 있습니다.

## 정답

① `\t` — 탭 문자
② `dropna` — 빈 값이 있는 행 빼기
③ `strip` — 앞뒤 공백 지우기
④ `drop_duplicates` — 겹치는 행 하나만 남기기
⑤ `transform` — 그룹 값을 행마다 붙이기
⑥ `lower` — 영문을 소문자로
⑦ `endswith` — 끝이 조사로 끝나면
⑧ `STOP` — 뺄 낱말 목록
⑨ `FILTER (WHERE label = 1)` — 긍정 리뷰만 세기
⑩ `pos + 1` — 0 을 없애는 평활
⑪ `pos + neg >= 50` — 50번 이상 나온 낱말만
⑫ `to_parquet` — Parquet 으로 저장